In [3]:
import pandas as pd
from pathlib import Path

old_folder = Path('dataframes/tagged_sentences_unordered_dfs')
new_folder = Path('dataframes/sentences_dfs')
out_folder = Path('dataframes/labeled_and_unlabled_sentences_dfs')
out_folder.mkdir(exist_ok=True)

def normalize(s):
    return ' '.join(str(s).split())

for old_path in old_folder.glob('*.csv'):
    new_path = new_folder / old_path.name
    if not new_path.exists():
        print(f"No match for {old_path.name}, skipping")
        continue

    old = pd.read_csv(old_path, header=None)
    old.columns = ['chapter_id', 'date', 'sentence'] + [f'label_{i}' for i in range(len(old.columns) - 3)]

    new = pd.read_csv(new_path, header=None)
    new.columns = ['book_name', 'chapter_id', 'entry_id', 'date', 'sentence']

    label_cols = [c for c in old.columns if c.startswith('label_')]

    old['sentence_norm'] = old['sentence'].apply(normalize)
    new['sentence_norm'] = new['sentence'].apply(normalize)

    merged = new.merge(
        old[['sentence_norm'] + label_cols],
        on='sentence_norm',
        how='left'
    ).drop(columns='sentence_norm')

    matched = merged[label_cols].notna().all(axis=1).sum()
    print(f"{old_path.stem}: {matched}/{len(new)} sentences matched")

    merged.to_csv(out_folder / old_path.name, index=False)

No match for AinUTL.csv, skipping
No match for DRtoF.csv, skipping
No match for PasWide.csv, skipping
No match for SofS.csv, skipping
No match for FitS.csv, skipping
No match for NbutC.csv, skipping
